In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
#set the seed
seed = 1234
# np.random.seed(seed)

In [2]:
import sys
print(sys.executable)
print(sys.version)

/usr/bin/python3
3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]


# Input data

In [3]:
#Input data
N = 5
L = 20
mu = np.array([1]*N)
sigma = np.array([1]*N)
pd=0.2 #probability of dying
ph=0.5 #probability of recovering
ps= 0.2 #probability of turning sick
d= 0.5  #distance of contagion




In [4]:
#Check of input data

# Classes

distance is $$\sqrt{r_1^2+r_2^2-2r_1 \cdot r_2}$$

In [5]:
class Population():
    def locate(self):
        self.position=np.random.uniform(0, self.L,(self.N,2)) #N*2 shape
        self.health=np.zeros(self.N, dtype=int) #N shape, 0=healthy, 1=sick, 2=dead
        self.health[0]=1 #first person is sick

    def __init__(self, N,L,  mu, sigma):
        self.N=N
        self.L=L
        self.mu=mu
        self.sigma=sigma
        self.contagions=[1, ]
        self.healings=[0, ]
        self.deaths=[0, ]
        self.locate()

    def move(self):
        rhos = np.random.normal(self.mu, self.sigma, (self.N)).reshape(self.N, 1)
        thetas = np.random.uniform(0, 2*np.pi, (self.N)) 
        self.position += rhos * np.array([np.cos(thetas), np.sin(thetas)]).T
        self.position = self.position % self.L        

    def compute_distances(self):
        """return matrix whose offdiagonal elements are distances squared between people"""
        rsquared=(self.position**2).sum(axis=1, keepdims=True)
       
        return rsquared+rsquared.T - 2 * self.position.dot(self.position.T) #N*N shape,
    def interact(self):

        prob=np.random.uniform(0, 1, (self.N))
        effective_prob = prob**self.health #prob for sick, 1 for healthy 
        mask_die = effective_prob < pd
        mask_recover = (pd <= effective_prob) & (effective_prob < (pd + ph))

        self.health[mask_die] += 1
        self.health[mask_recover] = 0
        self.deaths.append(np.sum(mask_die) + self.deaths[-1])
        self.healings.append(np.sum(mask_recover) + self.healings[-1])
        
        # print('Deaths today:', self.deaths[-1], 'Healings today:', self.healings[-1], 'Sick today:', self.contagions[-1], 'Alive today:', self.N-np.array(self.deaths).cumsum()[-1])
        # print("prob", prob, "effective_prob", effective_prob, "mask_die", mask_die, "mask_recover", mask_recover)

        ##getting sick
        self.distances = self.compute_distances()        
        max_distance=np.full((self.N, self.N), d*d)
        mask_close = (self.distances > 0) & (self.distances < max_distance) #it is like an adjacency matrix
        # print("mask_close", mask_close)
        # print("np.diag(self.health) ", self.health)


        mask_neighbor_sick = mask_close * self.health 

        # print("--->mask_neighbor_sick", mask_neighbor_sick)
        

        prob_sick=np.random.uniform(0, 1, (self.N, self.N))
        effective_prob_sick = prob_sick * mask_neighbor_sick #prob only where there is a sick neighbor

        mask_sick = ((effective_prob_sick < ps) & (effective_prob_sick > 0)).any(axis=1) #search in the columns, along the rows, for any sick neighbor, if yes, then True
        mask_sick = (mask_sick) & (self.health == 0) #only healthy can get sick

        # print(self.health)
        
        
        # print("mask_close", mask_close)
        # print("distances", self.distances)
        # print("prob_sick", prob_sick)
        # print("effective_prob_sick", effective_prob_sick)
        # print("mask_sick", mask_sick)


        self.health[mask_sick] = 1
        self.contagions.append(np.sum(mask_sick) + self.contagions[-1])
        # print(self.health)

    def dying(self):
        dying=np.where(self.health==2)
        # print('Dying today:', len(dying[0]))
        self.N=self.N-len(dying[0])
        self.health=np.delete(self.health, dying)
        self.position=np.delete(self.position, dying, axis=0)
        
        self.mu=np.delete(self.mu, dying)
        self.sigma=np.delete(self.sigma, dying)
    def a_day(self):
        self.move()
        self.interact()
        self.dying()
        # print("self.N", self.N)
        if self.N==0:
            print('All dead')
            return 1
        if self.N==1:
            print('All dead except one')
            return 1
        if self.health.sum()==0:
            print('All healthy')
            return 1
            

# Simulation

In [6]:

#Input data
N = 10000
L = 4
mu = np.array([1]*N)
sigma = np.array([1]*N)
pd=0.02 #probability of dying
ph=0.01 #probability of recovering
ps= 0.94 #probability of turning sick
d= L/5 #distance of contagion
d**2


0.6400000000000001

In [ ]:
pop=Population(N,L, mu, sigma)
for i in range(1000):
    ret=pop.a_day()
    if ret==1:
        break
# --- Plot variazioni giornaliere ---
contagions_arr = np.array(pop.contagions)
healings_arr   = np.array(pop.healings)
deaths_arr     = np.array(pop.deaths)

fig= plt.figure(figsize=(16, 8))
plt.subplot(1, 3, 1)
# --- Plot cumulativo ---
plt.plot(pop.contagions, label='Infected', alpha=0.5)
plt.plot(pop.healings,   label='Healed',   alpha=0.5)
plt.plot(pop.deaths,     label='Dead',     alpha=0.5)
plt.plot(contagions_arr-healings_arr-deaths_arr, label='Still infected', alpha=0.5)

plt.xlabel('Days')
plt.ylabel('Number of People')
plt.title('Epidemic Simulation - Cumulative')
plt.legend(), plt.grid()

# --- Plot variazioni giornaliere ---
contagions_arr = np.array(pop.contagions)
healings_arr   = np.array(pop.healings)
deaths_arr     = np.array(pop.deaths)


plt.subplot(1, 3, 2)
plt.plot(np.diff(contagions_arr), label='New Infected', alpha=0.5)
plt.plot(np.diff(healings_arr),   label='New Healed',   alpha=0.5)
plt.plot(np.diff(deaths_arr),     label='New Dead',     alpha=0.5)
plt.xlabel('Days')
plt.ylabel('Daily Change')
plt.title('Epidemic Simulation - Daily Variations')
plt.legend(), plt.grid()

# --- Plot alive (log scale) ---

plt.subplot(1, 3, 3)
plt.plot(N - deaths_arr, label='Alive', alpha=0.5)
plt.plot(contagions_arr-healings_arr-deaths_arr, label='Still infected', alpha=0.5)
plt.xlabel('Days')
plt.ylabel('Number of People')
plt.title('Alive (log scale)')
plt.legend(), plt.grid(), plt.show()

print(pop.N)